КТ5, Лазарева А.К., 3П3.24

In [1]:
!pip install httpx allure-pytest jsonschema pydantic pydantic-settings -q

Настройки

In [2]:
%%writefile settings.py
from pydantic_settings import BaseSettings


class Settings(BaseSettings):
    base_url: str = "https://petstore.swagger.io/v2"


base_settings = Settings()

Writing settings.py


Генерация тестовых данных

In [3]:
%%writefile fakers.py
from random import choice, randint
from string import ascii_letters, digits


def random_number(start: int = 100000, end: int = 999999) -> int:
    return randint(start, end)


def random_string(start: int = 9, end: int = 15) -> str:
    return "".join(choice(ascii_letters + digits) for _ in range(randint(start, end)))

Writing fakers.py


Модели Pydantic

In [4]:
%%writefile models.py
from pydantic import BaseModel, Field, RootModel

from fakers import random_number, random_string


class Category(BaseModel):
    id: int = Field(default_factory=random_number)
    name: str = "cats"


class Tag(BaseModel):
    id: int = Field(default_factory=random_number)
    name: str = Field(default_factory=random_string)


class Pet(BaseModel):
    id: int = Field(default_factory=random_number)
    category: Category = Field(default_factory=Category)
    name: str = "Муртузик"
    photo_urls: list[str] = Field(alias="photoUrls", default_factory=list)
    tags: list[Tag] = Field(default_factory=list)
    status: str = "available"


class Order(BaseModel):
    id: int = Field(default_factory=random_number)
    pet_id: int = Field(alias="petId", default_factory=random_number)
    quantity: int = Field(default_factory=lambda: random_number(1, 10))
    status: str = "placed"
    complete: bool = False


class Inventory(RootModel):
    root: dict[str, int]


class ApiResponse(BaseModel):
    code: int
    type: str
    message: str

Writing models.py


Базовый класс

In [5]:
%%writefile client.py
import time
from http import HTTPStatus

import allure
from httpx import Client


class HTTPClient(Client):
    def request(self, method, url, **kwargs):
        for _ in range(5):
            response = super().request(method, url, **kwargs)
            if response.status_code == HTTPStatus.OK:
                break
            time.sleep(1)
        return response

    @allure.step("GET {url}")
    def get(self, url, **kwargs):
        return super().get(url, **kwargs)

    @allure.step("POST {url}")
    def post(self, url, **kwargs):
        return super().post(url, **kwargs)

    @allure.step("PUT {url}")
    def put(self, url, **kwargs):
        return super().put(url, **kwargs)

    @allure.step("DELETE {url}")
    def delete(self, url, **kwargs):
        return super().delete(url, **kwargs)


class APIClient:
    def __init__(self, client: HTTPClient):
        self._client = client

    @property
    def client(self) -> HTTPClient:
        return self._client

Writing client.py


Клиенты для Pet и Store

In [6]:
%%writefile api.py
from enum import Enum

import allure
from httpx import Response

from client import APIClient
from models import Order, Pet


class APIRoutes(str, Enum):
    PET = "/pet"
    ORDER = "/store/order"
    INVENTORY = "/store/inventory"

    def __str__(self):
        return self.value


class PetClient(APIClient):
    @allure.step("Добавление питомца")
    def create_pet_api(self, payload: Pet) -> Response:
        return self.client.post(APIRoutes.PET, json=payload.model_dump(by_alias=True))

    @allure.step("Получение питомца с id {pet_id}")
    def get_pet_api(self, pet_id: int) -> Response:
        return self.client.get(f"{APIRoutes.PET}/{pet_id}")

    @allure.step("Изменение питомца")
    def update_pet_api(self, payload: Pet) -> Response:
        return self.client.put(APIRoutes.PET, json=payload.model_dump(by_alias=True))

    @allure.step("Удаление питомца с id {pet_id}")
    def delete_pet_api(self, pet_id: int) -> Response:
        return self.client.delete(f"{APIRoutes.PET}/{pet_id}")

    def create_pet(self) -> Pet:
        response = self.create_pet_api(Pet())
        return Pet(**response.json())


class StoreClient(APIClient):
    @allure.step("Создание заказа")
    def create_order_api(self, payload: Order) -> Response:
        return self.client.post(APIRoutes.ORDER, json=payload.model_dump(by_alias=True))

    @allure.step("Получение заказа с id {order_id}")
    def get_order_api(self, order_id: int) -> Response:
        return self.client.get(f"{APIRoutes.ORDER}/{order_id}")

    @allure.step("Получение остатков магазина")
    def get_inventory_api(self) -> Response:
        return self.client.get(APIRoutes.INVENTORY)

    @allure.step("Удаление заказа с id {order_id}")
    def delete_order_api(self, order_id: int) -> Response:
        return self.client.delete(f"{APIRoutes.ORDER}/{order_id}")

    def create_order(self) -> Order:
        response = self.create_order_api(Order())
        return Order(**response.json())

Writing api.py


Проверки

In [7]:
%%writefile assertions.py
import allure
from jsonschema import validate

from models import Order, Pet


@allure.step("Проверка статус кода")
def assert_status_code(actual: int, expected: int):
    assert actual == expected


@allure.step("Проверка JSON схемы")
def validate_schema(instance: dict, schema: dict):
    validate(instance=instance, schema=schema)


@allure.step("Проверка питомца")
def assert_pet(expected_pet: dict, actual_pet: Pet):
    assert expected_pet["id"] == actual_pet.id
    assert expected_pet["name"] == actual_pet.name
    assert expected_pet["category"]["name"] == actual_pet.category.name
    assert expected_pet["status"] == actual_pet.status


@allure.step("Проверка заказа")
def assert_order(expected_order: dict, actual_order: Order):
    assert expected_order["id"] == actual_order.id
    assert expected_order["petId"] == actual_order.pet_id
    assert expected_order["quantity"] == actual_order.quantity
    assert expected_order["status"] == actual_order.status

Writing assertions.py


Фикстуры

In [8]:
%%writefile conftest.py
import pytest

from api import PetClient, StoreClient
from client import HTTPClient
from models import Order, Pet
from settings import base_settings


@pytest.fixture(scope="class")
def class_pet_client() -> PetClient:
    return PetClient(client=HTTPClient(base_url=base_settings.base_url))


@pytest.fixture(scope="class")
def class_store_client() -> StoreClient:
    return StoreClient(client=HTTPClient(base_url=base_settings.base_url))


@pytest.fixture
def function_pet(class_pet_client: PetClient) -> Pet:
    pet = class_pet_client.create_pet()
    yield pet
    class_pet_client.delete_pet_api(pet.id)


@pytest.fixture
def function_order(class_store_client: StoreClient) -> Order:
    order = class_store_client.create_order()
    yield order
    class_store_client.delete_order_api(order.id)

Writing conftest.py


Pet

In [9]:
%%writefile test_pet.py
from http import HTTPStatus

import allure

from api import PetClient
from assertions import assert_pet, assert_status_code, validate_schema
from models import ApiResponse, Pet


@allure.feature("Pet")
@allure.story("Pet API")
class TestPet:
    @allure.title("Добавление питомца")
    def test_create_pet(self, class_pet_client: PetClient):
        payload = Pet()

        response = class_pet_client.create_pet_api(payload)

        assert_status_code(response.status_code, HTTPStatus.OK)
        assert_pet(response.json(), payload)
        validate_schema(response.json(), Pet.model_json_schema())

    @allure.title("Получение питомца по id")
    def test_get_pet(self, class_pet_client: PetClient, function_pet: Pet):
        response = class_pet_client.get_pet_api(function_pet.id)

        assert_status_code(response.status_code, HTTPStatus.OK)
        assert_pet(response.json(), function_pet)
        validate_schema(response.json(), Pet.model_json_schema())

    @allure.title("Изменение питомца")
    def test_update_pet(self, class_pet_client: PetClient, function_pet: Pet):
        function_pet.name = "Кутузик"
        function_pet.status = "sold"

        response = class_pet_client.update_pet_api(function_pet)

        assert_status_code(response.status_code, HTTPStatus.OK)
        assert_pet(response.json(), function_pet)
        validate_schema(response.json(), Pet.model_json_schema())

    @allure.title("Удаление питомца")
    def test_delete_pet(self, class_pet_client: PetClient, function_pet: Pet):
        delete_response = class_pet_client.delete_pet_api(function_pet.id)
        get_response = class_pet_client.get_pet_api(function_pet.id)

        assert_status_code(delete_response.status_code, HTTPStatus.OK)
        validate_schema(delete_response.json(), ApiResponse.model_json_schema())
        assert_status_code(get_response.status_code, HTTPStatus.NOT_FOUND)

Writing test_pet.py

In [10]:
!pytest test_pet.py --alluredir=allure-results --clean-alluredir -q

....                                                                     [100%]
4 passed in 14.13s


Store

In [11]:
%%writefile test_store.py
from http import HTTPStatus

import allure

from api import StoreClient
from assertions import assert_order, assert_status_code, validate_schema
from models import ApiResponse, Inventory, Order


@allure.feature("Store")
@allure.story("Store API")
class TestStore:
    @allure.title("Создание заказа")
    def test_create_order(self, class_store_client: StoreClient):
        payload = Order()

        response = class_store_client.create_order_api(payload)

        assert_status_code(response.status_code, HTTPStatus.OK)
        assert_order(response.json(), payload)
        validate_schema(response.json(), Order.model_json_schema())

    @allure.title("Получение заказа по id")
    def test_get_order(self, class_store_client: StoreClient, function_order: Order):
        response = class_store_client.get_order_api(function_order.id)

        assert_status_code(response.status_code, HTTPStatus.OK)
        assert_order(response.json(), function_order)
        validate_schema(response.json(), Order.model_json_schema())

    @allure.title("Получение остатков магазина")
    def test_get_inventory(self, class_store_client: StoreClient):
        response = class_store_client.get_inventory_api()

        assert_status_code(response.status_code, HTTPStatus.OK)
        validate_schema(response.json(), Inventory.model_json_schema())

    @allure.title("Удаление заказа")
    def test_delete_order(self, class_store_client: StoreClient, function_order: Order):
        delete_response = class_store_client.delete_order_api(function_order.id)
        get_response = class_store_client.get_order_api(function_order.id)

        assert_status_code(delete_response.status_code, HTTPStatus.OK)
        validate_schema(delete_response.json(), ApiResponse.model_json_schema())
        assert_status_code(get_response.status_code, HTTPStatus.NOT_FOUND)

Writing test_store.py


In [12]:
!pytest test_store.py --alluredir=allure-results -q

....                                                                     [100%]
4 passed in 14.40s


Отчёт Allure

In [13]:
!npx allure awesome allure-results --single-file

the report successfully generated (2004ms)
